# COMP560 Grader — Evaluation

Evaluates student-submitted prediction CSVs against ground truth pairs.parquet.

**Submission format:** `template_id_1, template_id_2, score`

## Configuration

In [ ]:
STUDENT_ID    = "YOUR_ID"
PREDICTION_PATH = "./predictions/dataset_a.csv"  # path to CSV or directory of CSVs
DATASETS_ROOT = "./datasets"
OUTPUT_DIR    = "./results"
DATASETS      = ["dataset_a", "dataset_b"]  # which datasets to evaluate

In [ ]:
import json
import os
from datetime import datetime
from pathlib import Path
from typing import Dict, List

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve, auc
from tqdm import tqdm

## Metric Functions

In [ ]:
def compute_tar_at_far(scores, labels, far_targets=[1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]):
    fpr, tpr, _ = roc_curve(labels, scores)
    results = {}
    for far in far_targets:
        idx = np.argmin(np.abs(fpr - far))
        results[f"TAR@FAR={far:.0e}"] = tpr[idx] * 100
    results["AUC"] = auc(fpr, tpr) * 100
    return results

## Dataset Evaluation

In [ ]:
def evaluate_dataset(prediction_path, gt_pairs_path, dataset_name):
    gt_df = pd.read_parquet(gt_pairs_path)
    gt_df["pair_key"] = gt_df["template_id_1"].astype(str) + "_" + gt_df["template_id_2"].astype(str)

    pred_df = pd.read_csv(prediction_path)
    required_cols = {"template_id_1", "template_id_2", "score"}
    if not required_cols.issubset(pred_df.columns):
        raise ValueError(f"Prediction CSV must have columns: {required_cols}. Got: {set(pred_df.columns)}")
    pred_df["pair_key"] = pred_df["template_id_1"].astype(str) + "_" + pred_df["template_id_2"].astype(str)

    merged = gt_df.merge(pred_df[["pair_key", "score"]], on="pair_key", how="left")
    missing = merged["score"].isna().sum()
    if missing > 0:
        print(f"  WARNING: {missing}/{len(merged)} pairs have no prediction (will use score=0)")
        merged["score"] = merged["score"].fillna(0.0)

    scores = merged["score"].values
    labels = merged["label"].values
    performance = compute_tar_at_far(scores, labels)

    return {
        "performance": performance,
        "submission_info": {
            "num_predicted_pairs": len(pred_df),
            "num_gt_pairs": len(gt_df),
            "num_matched_pairs": len(merged) - missing,
            "num_missing_pairs": int(missing),
            "num_positive_pairs": int(labels.sum()),
            "num_negative_pairs": int((labels == 0).sum()),
        },
    }

## Run Evaluation

In [ ]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

print("=" * 60)
print("COMP560 Face Recognition Evaluation")
print("=" * 60)
print(f"Student ID: {STUDENT_ID}")
print(f"Prediction: {PREDICTION_PATH}")

all_results = {
    "student_id": STUDENT_ID,
    "timestamp": timestamp,
    "prediction_path": PREDICTION_PATH,
    "datasets": {},
}

for dataset_name in DATASETS:
    print(f"\n{'=' * 60}")
    print(f"Evaluating on {dataset_name}")
    gt_pairs_path = os.path.join(DATASETS_ROOT, dataset_name, "pairs.parquet")
    if not os.path.exists(gt_pairs_path):
        print(f"  ERROR: Ground truth not found at {gt_pairs_path}")
        all_results["datasets"][dataset_name] = {"error": "ground truth not found"}
        continue

    pred_path = os.path.join(PREDICTION_PATH, f"{dataset_name}.csv") if os.path.isdir(PREDICTION_PATH) else PREDICTION_PATH
    if not os.path.exists(pred_path):
        print(f"  ERROR: Prediction file not found at {pred_path}")
        all_results["datasets"][dataset_name] = {"error": "prediction file not found"}
        continue

    try:
        results = evaluate_dataset(pred_path, gt_pairs_path, dataset_name)
        all_results["datasets"][dataset_name] = results
        print("\nPerformance Metrics (TAR@FAR):")
        for metric, value in results["performance"].items():
            print(f"  {metric}: {value:.2f}%")
        print("\nSubmission Info:")
        for key, value in results["submission_info"].items():
            print(f"  {key}: {value}")
    except Exception as e:
        import traceback
        print(f"  ERROR: {e}")
        traceback.print_exc()
        all_results["datasets"][dataset_name] = {"error": str(e)}

## Save Results

In [ ]:
output_file = output_dir / f"{STUDENT_ID}_{timestamp}.json"
with open(output_file, "w") as f:
    json.dump(all_results, f, indent=2)
print(f"Results saved to: {output_file}")

summary_file = output_dir / f"{STUDENT_ID}_{timestamp}_summary.csv"
with open(summary_file, "w") as f:
    f.write("dataset,TAR@FAR=1e-6,TAR@FAR=1e-5,TAR@FAR=1e-4,TAR@FAR=1e-3,AUC\n")
    for dataset_name, results in all_results["datasets"].items():
        if "error" not in results:
            perf = results["performance"]
            f.write(
                f"{dataset_name},"
                f"{perf.get('TAR@FAR=1e-06', 0):.2f},"
                f"{perf.get('TAR@FAR=1e-05', 0):.2f},"
                f"{perf.get('TAR@FAR=1e-04', 0):.2f},"
                f"{perf.get('TAR@FAR=1e-03', 0):.2f},"
                f"{perf.get('AUC', 0):.2f}\n"
            )
print(f"Summary saved to: {summary_file}")